In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import random
import numpy as np
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from torch.nn import CrossEntropyLoss
from transformers import Trainer

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
df_train = pd.read_csv('/kaggle/input/datasets/suruadelina/liar-2/train.csv', encoding="utf-8")
df_test = pd.read_csv('/kaggle/input/datasets/suruadelina/liar-2/test.csv', encoding="utf-8")
df_val = pd.read_csv('/kaggle/input/datasets/suruadelina/liar-2/valid.csv', encoding="utf-8")
df_aug = pd.read_csv('/kaggle/input/datasets/suruadelina/augmentations/augmented_dataset.csv', encoding="utf-8")

In [5]:
def clean(text):
    if pd.isna(text):
        return text
    text = text.lower()
    text = text.strip()
    text = re.sub(r'https\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [6]:
label_mapping = {
    0: 0,  # Pants on Fire → Fake
    1: 0,  # False → Fake
    2: 0,  # Barely True → Fake
    3: 1,  # Half True → True
    4: 1,  # Mostly True → True
    5: 1   # True → True
}

In [7]:
df_train['label'] = df_train['label'].map(label_mapping)
df_val['label'] = df_val['label'].map(label_mapping)
df_test['label'] = df_test['label'].map(label_mapping)
df_aug['label'] = df_aug['label'].map(label_mapping)
print(df_train['label'].value_counts())

label
0    10591
1     7778
Name: count, dtype: int64


In [8]:
df_train['statement'] = df_train['statement'].apply(clean)
df_test['statement'] = df_test['statement'].apply(clean)
df_val['statement'] = df_val['statement'].apply(clean)

In [9]:
df_train = df_train[['statement', 'label']]
df_test = df_test[['statement', 'label']]
df_val = df_val[['statement', 'label']]

In [10]:
df_aug.columns

Index(['statement', 'label', 'statement_aug_de', 'statement_aug_et',
       'statement_aug_ar', 'statement_aug_zh', 'statement_aug_ru',
       'statement_aug_af', 'statement_aug_az', 'statement_aug_bem',
       'statement_aug_ca', 'statement_aug_fr'],
      dtype='object')

In [11]:
df_train.shape

(18369, 2)

In [12]:
df_train = pd.concat([df_train, df_aug[['statement_aug_de', 'label']].rename(columns={"statement_aug_de": "statement"})], ignore_index=True)
df_train = pd.concat([df_train, df_aug[['statement_aug_zh', 'label']].rename(columns={"statement_aug_zh": "statement"})], ignore_index=True)
df_train = pd.concat([df_train, df_aug[['statement_aug_ru', 'label']].rename(columns={"statement_aug_ru": "statement"})], ignore_index=True)

In [13]:
df_train.shape

(73476, 2)

In [14]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
max_length = 512

train_encodings = tokenizer(
    df_train['statement'].to_list(),
    padding=True,
    truncation=True,
    max_length=max_length
)
test_encodings = tokenizer(
    df_test['statement'].to_list(), 
    padding=True, 
    truncation=True, 
    max_length=max_length
)

val_encodings = tokenizer(
    df_val['statement'].to_list(),
    padding=True,
    truncation=True,
    max_length=max_length
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [15]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        print({k: len(v) for k, v in self.encodings.items()})
        print(len(self.labels))

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [16]:
train_dataset = MyDataset(train_encodings, df_train['label'].to_list())
test_dataset = MyDataset(test_encodings, df_test['label'].to_list())
val_dataset = MyDataset(val_encodings, df_val['label'].to_list())

{'input_ids': 73476, 'attention_mask': 73476}
73476
{'input_ids': 2296, 'attention_mask': 2296}
2296
{'input_ids': 2297, 'attention_mask': 2297}
2297


In [17]:
model = AutoModelForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
from sklearn.metrics import roc_auc_score, f1_score
import numpy as np
import torch

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(logits, axis=1)

    acc = (preds == labels).mean()

    f1 = f1_score(labels, preds, average="macro")

    auc = roc_auc_score(labels, probs[:, 1])  

    return {
        "accuracy": acc,
        "f1": f1,
        "auc": auc
    }

In [19]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/output",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=1e-5,        
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,            
    weight_decay=0.01,
    seed=SEED,
    dataloader_num_workers=0,
    report_to='none',
    logging_strategy='steps',
    logging_steps=1,
    logging_first_step=True,
)

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Auc
1,0.365881,0.561116,0.720070,0.709507,0.795051
2,0.076105,0.716241,0.710057,0.703755,0.784795
3,0.284241,0.971108,0.710492,0.708145,0.779067


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=13779, training_loss=0.40859182889169293, metrics={'train_runtime': 12033.3791, 'train_samples_per_second': 30.53, 'train_steps_per_second': 1.908, 'total_flos': 5.799704371089408e+16, 'train_loss': 0.40859182889169293, 'epoch': 3.0})

In [22]:
predictions = trainer.predict(test_dataset)

final_metrics = compute_metrics(
    (predictions.predictions, predictions.label_ids)
)

print("\nFinal Evaluation:")
print(final_metrics)


Final Evaluation:
{'accuracy': np.float64(0.705574912891986), 'f1': 0.6948726840657116, 'auc': np.float64(0.7914515812034532)}


In [23]:
trainer.save_model('/kaggle/working/BERT_aug_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]